# Opentrons Temperature Module

The Opentrons Temperature Module GEN2 is a Peltier-based temperature controller for microplates and tube racks. It supports heating (up to 95 °C) and active cooling (down to 4 °C).

| Model | PLR Name | Temperature Range | Communication |
|---|---|---|---|
| Temperature Module GEN2 | `OpentronsTemperatureModuleV2` | 4--95 °C | USB via OT-2 HTTP API or direct USB serial |

The module can be used in two modes:

- **Via the Opentrons robot** -- communicates through the OT-2 HTTP API (requires `ot_api`).
- **Via direct USB serial** -- communicates over a serial port, no OT-2 robot required.

See the [Opentrons product page](https://opentrons.com/products/temperature-module-gen2) for hardware details.

## Setup (via Opentrons robot)

When using the module through an OT-2, first set up the liquid handler and then list connected modules to find the module ID.

In [ ]:
from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends.opentrons_backend import OpentronsBackend
from pylabrobot.resources.opentrons import OTDeck

ot = OpentronsBackend(host="169.254.184.185", port=31950)  # replace with your robot's IP
lh = LiquidHandler(backend=ot, deck=OTDeck())
await lh.setup()

# List connected modules to find the temperature module ID
await ot.list_connected_modules()

In [ ]:
from pylabrobot.opentrons import OpentronsTemperatureModuleV2

t = OpentronsTemperatureModuleV2(
    name="t",
    opentrons_id="fc409cc91770129af8eb0a01724c56cb052b306a",  # from list_connected_modules()
)
await t.setup()

# Assign the module to a deck slot
lh.deck.assign_child_at_slot(t, slot=3)

## Setup (via direct USB serial)

If you are not using an OT-2 robot, connect the temperature module directly to your computer via USB and specify the serial port.

In [ ]:
from pylabrobot.opentrons import OpentronsTemperatureModuleV2

t = OpentronsTemperatureModuleV2(name="t", serial_port="/dev/ttyACM0")  # replace with your port
await t.setup()

## Temperature control

The module exposes a {class}`~pylabrobot.capabilities.temperature_controlling.temperature_controller.TemperatureController` on `t.tc`. For the full API, see [Temperature Control](../../capabilities/temperature-control).

In [ ]:
await t.tc.set_temperature(37.0)
await t.tc.wait_for_temperature()

In [ ]:
current = await t.tc.request_temperature()
print(f"{current:.1f} \u00b0C")

In [ ]:
await t.tc.deactivate()

## Pipetting from the temperature module

When using the module on an OT-2 deck, you can pipette directly to/from its child resource (e.g. a tube rack or well plate). Assign the child resource when creating the module, or afterwards.

In [ ]:
from pylabrobot.resources.opentrons import opentrons_96_tiprack_300ul

tips300 = opentrons_96_tiprack_300ul(name="tips")
lh.deck.assign_child_at_slot(tips300, slot=11)

await lh.pick_up_tips(tips300["A5"])
await lh.aspirate(t.resource["A1"], vols=[20])
await lh.return_tips()

## Teardown

In [ ]:
await t.stop()